<a href="https://colab.research.google.com/github/blancavazquez/PLN/blob/2027-01/notebooks/4_4_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BERT

El objetivo de esta libreta es usar el modelo BERT en diferentes tareas.

# Tokenizando textos

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-cased")
text = 'UNAM: It was founded on September 21, 1551, under the name of the Royal and Pontifical University of Mexico. It is the largest and most important university in Mexico. Throughout time, it has played a leading role in the history and formation of our country. UNAM is recognized as a space of freedom. Respect, tolerance, and dialogue are practiced. The plurality of ideas and thought is appreciated as a sign of its wealth.'
 ## Tokenize and encode the text
encoding = tokenizer.encode(text)
print("Token IDs:", encoding)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Token IDs: [101, 7414, 10964, 131, 1135, 1108, 1771, 1113, 1347, 1626, 117, 14691, 1475, 117, 1223, 1103, 1271, 1104, 1103, 1787, 1105, 27164, 1239, 1104, 2470, 119, 1135, 1110, 1103, 2026, 1105, 1211, 1696, 2755, 1107, 2470, 119, 7092, 1159, 117, 1122, 1144, 1307, 170, 2020, 1648, 1107, 1103, 1607, 1105, 3855, 1104, 1412, 1583, 119, 7414, 10964, 1110, 3037, 1112, 170, 2000, 1104, 4438, 119, 11336, 20629, 117, 15745, 117, 1105, 8556, 1132, 8720, 119, 1109, 14920, 1785, 1104, 4133, 1105, 1354, 1110, 12503, 1112, 170, 2951, 1104, 1157, 6968, 119, 102]


In [ ]:
# Convert token IDs back to tokens
tokens = tokenizer.convert_ids_to_tokens(encoding)
print(f"Original text: {text}")
print("Tokens:", tokens)

Original text: UNAM: It was founded on September 21, 1551, under the name of the Royal and Pontifical University of Mexico. It is the largest and most important university in Mexico. Throughout time, it has played a leading role in the history and formation of our country. UNAM is recognized as a space of freedom. Respect, tolerance, and dialogue are practiced. The plurality of ideas and thought is appreciated as a sign of its wealth.
Tokens: ['[CLS]', 'UN', '##AM', ':', 'It', 'was', 'founded', 'on', 'September', '21', ',', '155', '##1', ',', 'under', 'the', 'name', 'of', 'the', 'Royal', 'and', 'Pontifical', 'University', 'of', 'Mexico', '.', 'It', 'is', 'the', 'largest', 'and', 'most', 'important', 'university', 'in', 'Mexico', '.', 'Throughout', 'time', ',', 'it', 'has', 'played', 'a', 'leading', 'role', 'in', 'the', 'history', 'and', 'formation', 'of', 'our', 'country', '.', 'UN', '##AM', 'is', 'recognized', 'as', 'a', 'space', 'of', 'freedom', '.', 'Re', '##spect', ',', 'tolerance'

# Análisis de sentimientos

In [ ]:
import torch
from transformers import pipeline

# Create a sentiment analysis pipeline
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# Test text
text = "me duele la panza."

# Get the sentiment
result = sentiment_analyzer(text)
print(f"Sentiment: {result[0]['label']}")
print(f"Confidence: {result[0]['score']:.4f}")

Device set to use cpu


Sentiment: NEGATIVE
Confidence: 0.9387


# Análisis de sentimientos (sin pipeline)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

class BERTSentimentAnalyzer:
    def __init__(self, model_name="distilbert-base-uncased-finetuned-sst-2-english"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval()
        self.labels = ['NEGATIVE', 'POSITIVE']

    def preprocess_text(self, text):
        # Remove extra whitespace and normalize
        text = ' '.join(text.split())

        # Tokenize with BERT-specific tokens
        inputs = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=512,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        # Move to GPU if available
        return {k: v.to(self.device) for k, v in inputs.items()}

    def predict(self, text):
        # Prepare text for model
        inputs = self.preprocess_text(text)

        # Get model predictions
        with torch.no_grad():
            outputs = self.model(**inputs)
            probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)

        # Convert to human-readable format
        prediction_dict = {
            'text': text,
            'sentiment': self.labels[probabilities.argmax().item()],
            'confidence': probabilities.max().item(),
            'probabilities': {
                label: prob.item()
                for label, prob in zip(self.labels, probabilities[0])
            }
        }
        return prediction_dict

In [ ]:
def demonstrate_sentiment_analysis():
    # Initialize analyzer
    analyzer = BERTSentimentAnalyzer()

    # Test texts
    texts = [
        "No dormí bien",
        "Good experience, but I did not recommend.",
        "El día está nublado."
    ]

    # Analyze each text
    for text in texts:
        result = analyzer.predict(text)
        print(f"\nText: {result['text']}")
        print(f"Sentiment: {result['sentiment']}")
        print(f"Confidence: {result['confidence']:.4f}")
        print("Detailed probabilities:")
        for label, prob in result['probabilities'].items():
            print(f"  {label}: {prob:.4f}")

# Running demonstration
demonstrate_sentiment_analysis()


Text: No dormí bien
Sentiment: NEGATIVE
Confidence: 0.9938
Detailed probabilities:
  NEGATIVE: 0.9938
  POSITIVE: 0.0062

Text: Good experience, but I did not recommend.
Sentiment: NEGATIVE
Confidence: 0.6913
Detailed probabilities:
  NEGATIVE: 0.6913
  POSITIVE: 0.3087

Text: El día está nublado.
Sentiment: POSITIVE
Confidence: 0.9840
Detailed probabilities:
  NEGATIVE: 0.0160
  POSITIVE: 0.9840


# Reconocimiento de entidades (NER)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

class BERTNamedEntityRecognizer:
    def __init__(self):
        self.tokenizer = AutoTokenizer.from_pretrained("dslim/bert-base-NER")
        self.model = AutoModelForTokenClassification.from_pretrained("dslim/bert-base-NER")
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval()

    def recognize_entities(self, text):
        # Tokenize input text
        inputs = self.tokenizer(
            text,
            add_special_tokens=True,
            return_tensors="pt",
            padding=True,
            truncation=True
        )

        # Move inputs to device
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        # print(inputs)

        # Get predictions
        with torch.no_grad():
            outputs = self.model(**inputs)
            predictions = outputs.logits.argmax(-1)

        # Convert predictions to entities
        tokens = self.tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
        labels = [self.model.config.id2label[p.item()] for p in predictions[0]]
        # print(labels)

        # Extract entities
        entities = []
        current_entity = None

        for token, label in zip(tokens, labels):
            if label.startswith('B-'):
                if current_entity:
                    entities.append(current_entity)
                current_entity = {'type': label[2:], 'text': token}
            elif label.startswith('I-') and current_entity:
                if token.startswith('##'):
                    current_entity['text'] += token[2:]
                else:
                    current_entity['text'] += ' ' + token
            elif label == 'O':
                if current_entity:
                    entities.append(current_entity)
                    current_entity = None

        if current_entity:
            entities.append(current_entity)

        return entities

In [ ]:
def demonstrate_ner():
    # Initialize recognizer
    ner = BERTNamedEntityRecognizer()

    # Example text
    text = """
    Panificación Bimbo inició operaciones el 2 de diciembre de 1945, México, unam, playa, uady, yucatan, yucatán, YUCATÁN"
    """

    # Get entities
    entities = ner.recognize_entities(text)

    # Display results
    print("Found entities:")
    for entity in entities:
        print(f"- {entity['text']} ({entity['type']})")

# Running demonstration
demonstrate_ner()

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Found entities:
- Panificación Bimbo (ORG)
- México (LOC)
- YUCAT (LOC)
